# Advanced Stock Market Analysis Dashboard
## 2024 – Present  |  Technical · Statistical · Machine Learning

---

**Stocks Covered:** `TSLA` · `RIVN` · `NIO` · `LCID` · `GM` · `F` · `SPY`  
**Data Source:** Yahoo Finance via `yfinance` (live, auto-updating)  
**Period:** January 2024 – Present

| # | Section |
|---|---|
| 1 | Data Acquisition & Validation |
| 2 | Exploratory Data Analysis |
| 3 | Technical Indicators (SMA · EMA · BB · RSI · MACD · ATR) |
| 4 | Returns Distribution & Rolling Volatility |
| 5 | Risk Metrics (Sharpe · VaR · CVaR · Max Drawdown) |
| 6 | Multi-Stock Comparative Analysis |
| 7 | ML Price Prediction (Random Forest) |
| 8 | Backtesting: MA Crossover Strategy |
| 9 | Summary Dashboard |

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns
import yfinance as yf
import warnings
import datetime

from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({
    'figure.figsize': (16, 6),
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
})
%matplotlib inline

print(f'Libraries loaded. Analysis date: {datetime.date.today()}')

---
## 1. Data Acquisition (2024 – Present)
Pulling live daily OHLCV data from Yahoo Finance for 7 tickers.

In [ ]:
START_DATE = '2024-01-01'
END_DATE   = datetime.date.today().strftime('%Y-%m-%d')
FOCUS      = 'TSLA'   # primary ticker for deep-dive analysis

STOCKS = {
    'TSLA': 'Tesla Inc.',
    'RIVN': 'Rivian Automotive',
    'NIO' : 'NIO Inc.',
    'LCID': 'Lucid Group',
    'GM'  : 'General Motors',
    'F'   : 'Ford Motor',
    'SPY' : 'S&P 500 ETF',
}

print(f'Fetching {len(STOCKS)} stocks: {START_DATE} → {END_DATE}\n')
stock_data = {}
for ticker, name in STOCKS.items():
    df = yf.download(ticker, start=START_DATE, end=END_DATE,
                     progress=False, auto_adjust=True)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    if not df.empty:
        stock_data[ticker] = df
        latest_close = float(df['Close'].iloc[-1])
        print(f'  {ticker:<5} {name:<22} {len(df):>3} days | ${latest_close:>8.2f}')

# Combined closing-price DataFrame
close = pd.DataFrame({t: stock_data[t]['Close'].squeeze() for t in stock_data})
print(f'\nDate range: {close.index[0].date()} → {close.index[-1].date()}')
print(f'Shape: {close.shape}')

In [ ]:
# Year-to-date return from Jan 1 2024
ytd_return = (close.iloc[-1] / close.iloc[0] - 1).sort_values(ascending=False)

print('=== Performance Since Jan 1 2024 ===')
for t, r in ytd_return.items():
    bar = '█' * int(abs(r * 20)) if abs(r) < 5 else '█' * 100
    sign = '+' if r >= 0 else ''
    print(f'  {t:<5} {sign}{r:>7.1%}  {bar[:30]}')

print('\nDescriptive Statistics (Close Prices):')
close.describe().round(2)

---
## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=False)

# Panel 1: absolute prices (log scale for multi-magnitude stocks)
for t in close.columns:
    axes[0].plot(close.index, close[t], lw=1.4, label=t)
axes[0].set_yscale('log')
axes[0].set_title('Closing Prices (Log Scale) – 2024 to Present')
axes[0].set_ylabel('Price (USD, log)')
axes[0].legend(ncol=4, fontsize=9)

# Panel 2: normalised to 100
norm = close.div(close.iloc[0]) * 100
for t in norm.columns:
    axes[1].plot(norm.index, norm[t], lw=1.4, label=t)
axes[1].axhline(100, color='black', ls='--', lw=0.8, alpha=0.5)
axes[1].set_title('Normalised Performance (Base = 100)')
axes[1].set_ylabel('Indexed Price')
axes[1].legend(ncol=4, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
tsla_raw = stock_data[FOCUS].copy()

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

# Price
axes[0].plot(tsla_raw.index, tsla_raw['Close'], lw=1.5, color='#1f77b4', label='Close')
axes[0].fill_between(tsla_raw.index, tsla_raw['Low'], tsla_raw['High'],
                     alpha=0.15, color='#1f77b4', label='High-Low range')
axes[0].set_title(f'{FOCUS} Price History 2024–Present')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(fontsize=9)

# Volume coloured by direction
ret = tsla_raw['Close'].pct_change()
vol_colors = ['#2ca02c' if r >= 0 else '#d62728' for r in ret.fillna(0)]
axes[1].bar(tsla_raw.index, tsla_raw['Volume'], color=vol_colors, alpha=0.7, width=1)
axes[1].set_title('Trading Volume (green = up day)')
axes[1].set_ylabel('Volume')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

plt.tight_layout()
plt.show()

---
## 3. Technical Indicators

Indicators computed:
- **Trend:** SMA 20/50/200, EMA 12/26
- **Bands:** Bollinger Bands (20, ±2σ)
- **Momentum:** RSI (14), MACD (12/26/9)
- **Volatility:** ATR (14), Rolling Volatility (20-day annualised)
- **Volume:** On-Balance Volume (OBV)

In [ ]:
def add_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Add comprehensive technical indicators to an OHLCV DataFrame."""
    d = df.copy()
    c, h, l, v = d['Close'], d['High'], d['Low'], d['Volume']

    # ── Moving Averages ────────────────────────────────────────────────────
    d['SMA_20']  = c.rolling(20).mean()
    d['SMA_50']  = c.rolling(50).mean()
    d['SMA_200'] = c.rolling(200).mean()
    d['EMA_12']  = c.ewm(span=12, adjust=False).mean()
    d['EMA_26']  = c.ewm(span=26, adjust=False).mean()

    # ── Bollinger Bands ────────────────────────────────────────────────────
    mid = c.rolling(20).mean()
    std = c.rolling(20).std()
    d['BB_Upper'] = mid + 2 * std
    d['BB_Mid']   = mid
    d['BB_Lower'] = mid - 2 * std
    d['BB_Width'] = (d['BB_Upper'] - d['BB_Lower']) / mid
    d['BB_Pct']   = (c - d['BB_Lower']) / (d['BB_Upper'] - d['BB_Lower'])

    # ── RSI (14) ───────────────────────────────────────────────────────────
    delta     = c.diff()
    gain      = delta.clip(lower=0).ewm(com=13, adjust=False).mean()
    loss      = (-delta.clip(upper=0)).ewm(com=13, adjust=False).mean()
    d['RSI']  = 100 - 100 / (1 + gain / loss)

    # ── MACD ──────────────────────────────────────────────────────────────
    d['MACD']        = d['EMA_12'] - d['EMA_26']
    d['MACD_Signal'] = d['MACD'].ewm(span=9, adjust=False).mean()
    d['MACD_Hist']   = d['MACD'] - d['MACD_Signal']

    # ── ATR (14) ──────────────────────────────────────────────────────────
    tr = pd.concat([
        h - l,
        (h - c.shift()).abs(),
        (l - c.shift()).abs()
    ], axis=1).max(axis=1)
    d['ATR'] = tr.rolling(14).mean()

    # ── OBV ───────────────────────────────────────────────────────────────
    direction = np.sign(c.diff().fillna(0))
    d['OBV']  = (direction * v).cumsum()

    # ── Returns & Volatility ───────────────────────────────────────────────
    d['Return']     = c.pct_change()
    d['Log_Return'] = np.log(c / c.shift())
    d['Cum_Return'] = (1 + d['Return']).cumprod() - 1
    d['Vol_20']     = d['Return'].rolling(20).std() * np.sqrt(252)
    d['Vol_60']     = d['Return'].rolling(60).std() * np.sqrt(252)

    # ── Price vs MA ratio (useful for ML) ─────────────────────────────────
    d['Ratio_SMA20'] = c / d['SMA_20']
    d['Ratio_SMA50'] = c / d['SMA_50']

    return d

# Apply to all stocks
ind_data = {t: add_indicators(stock_data[t]) for t in stock_data}
tsla = ind_data[FOCUS]

new_cols = [c for c in tsla.columns if c not in stock_data[FOCUS].columns]
print(f'Indicators added: {new_cols}')
print(f'\nTSLA latest row:\n{tsla[new_cols].iloc[-1].round(3)}')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10),
                         gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax = axes[0]
ax.plot(tsla.index, tsla['Close'],   lw=1.8, color='#1f77b4', label='Close', zorder=3)
ax.plot(tsla.index, tsla['SMA_20'],  lw=1.2, color='orange',  label='SMA 20', alpha=0.9)
ax.plot(tsla.index, tsla['SMA_50'],  lw=1.2, color='green',   label='SMA 50', alpha=0.9)
ax.plot(tsla.index, tsla['SMA_200'], lw=1.2, color='red',     label='SMA 200', alpha=0.9)
ax.fill_between(tsla.index, tsla['BB_Lower'], tsla['BB_Upper'],
                alpha=0.12, color='purple', label='Bollinger Bands (20, ±2σ)')
ax.plot(tsla.index, tsla['BB_Upper'], lw=0.7, color='purple', ls='--', alpha=0.5)
ax.plot(tsla.index, tsla['BB_Lower'], lw=0.7, color='purple', ls='--', alpha=0.5)
ax.set_title(f'{FOCUS} – Price, Moving Averages & Bollinger Bands')
ax.set_ylabel('Price (USD)')
ax.legend(ncol=6, fontsize=9)

# Bollinger Band Width
axes[1].plot(tsla.index, tsla['BB_Width'], color='purple', lw=1.2)
axes[1].fill_between(tsla.index, tsla['BB_Width'], alpha=0.2, color='purple')
axes[1].set_title('Bollinger Band Width (volatility proxy)')
axes[1].set_ylabel('BB Width')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True,
                         gridspec_kw={'height_ratios': [3, 2, 2]})

# Price
axes[0].plot(tsla.index, tsla['Close'], lw=1.5, color='#1f77b4')
axes[0].set_title(f'{FOCUS} Close Price')
axes[0].set_ylabel('Price (USD)')

# RSI
axes[1].plot(tsla.index, tsla['RSI'], color='#9467bd', lw=1.2, label='RSI (14)')
axes[1].axhline(70, color='red',   ls='--', lw=1, alpha=0.7)
axes[1].axhline(50, color='gray',  ls=':',  lw=0.8, alpha=0.5)
axes[1].axhline(30, color='green', ls='--', lw=1, alpha=0.7)
axes[1].fill_between(tsla.index, tsla['RSI'], 70,
                     where=(tsla['RSI'] >= 70), alpha=0.2, color='red',   label='Overbought')
axes[1].fill_between(tsla.index, tsla['RSI'], 30,
                     where=(tsla['RSI'] <= 30), alpha=0.2, color='green', label='Oversold')
axes[1].set_ylim(0, 100)
axes[1].set_title('RSI (14)')
axes[1].set_ylabel('RSI')
axes[1].legend(fontsize=9)

# MACD
axes[2].plot(tsla.index, tsla['MACD'],        color='blue',   lw=1.2, label='MACD')
axes[2].plot(tsla.index, tsla['MACD_Signal'], color='orange', lw=1.2, label='Signal (9)')
hist_colors = ['#2ca02c' if v >= 0 else '#d62728'
               for v in tsla['MACD_Hist'].fillna(0)]
axes[2].bar(tsla.index, tsla['MACD_Hist'], color=hist_colors, alpha=0.5, width=1)
axes[2].axhline(0, color='black', lw=0.8, ls='--')
axes[2].set_title('MACD (12, 26, 9)')
axes[2].set_ylabel('MACD')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# ATR
axes[0, 0].plot(tsla.index, tsla['ATR'], color='#e377c2', lw=1.2)
axes[0, 0].fill_between(tsla.index, tsla['ATR'], alpha=0.2, color='#e377c2')
axes[0, 0].set_title(f'{FOCUS} ATR (14) – Average True Range')
axes[0, 0].set_ylabel('ATR (USD)')

# OBV
axes[0, 1].plot(tsla.index, tsla['OBV'] / 1e9, color='#17becf', lw=1.2)
axes[0, 1].fill_between(tsla.index, tsla['OBV'] / 1e9, alpha=0.15, color='#17becf')
axes[0, 1].set_title(f'{FOCUS} On-Balance Volume (OBV)')
axes[0, 1].set_ylabel('OBV (billions)')

# Rolling Volatility (annualised)
axes[1, 0].plot(tsla.index, tsla['Vol_20'], lw=1.2, label='20-day', color='orange')
axes[1, 0].plot(tsla.index, tsla['Vol_60'], lw=1.2, label='60-day', color='red')
axes[1, 0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1, 0].set_title(f'{FOCUS} Rolling Annualised Volatility')
axes[1, 0].set_ylabel('Volatility')
axes[1, 0].legend(fontsize=9)

# BB %B
axes[1, 1].plot(tsla.index, tsla['BB_Pct'], color='purple', lw=1.2, label='%B')
axes[1, 1].axhline(1.0, color='red',   ls='--', lw=1, alpha=0.7, label='Upper band')
axes[1, 1].axhline(0.5, color='gray',  ls=':',  lw=0.8)
axes[1, 1].axhline(0.0, color='green', ls='--', lw=1, alpha=0.7, label='Lower band')
axes[1, 1].set_title(f'{FOCUS} Bollinger %B')
axes[1, 1].set_ylabel('%B')
axes[1, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 4. Returns Distribution & Rolling Volatility

In [ ]:
daily_returns = close.pct_change().dropna()
tsla_ret      = daily_returns[FOCUS]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Histogram + normal fit
axes[0].hist(tsla_ret, bins=70, density=True, color='#1f77b4',
             edgecolor='white', alpha=0.8, label='Empirical')
mu, sigma = float(tsla_ret.mean()), float(tsla_ret.std())
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
axes[0].plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label='Normal fit')
axes[0].axvline(np.percentile(tsla_ret, 5),  color='orange', ls='--', lw=1.5, label='VaR 95%')
axes[0].axvline(np.percentile(tsla_ret, 1),  color='red',    ls='--', lw=1.5, label='VaR 99%')
axes[0].set_title(f'{FOCUS} Daily Returns Distribution')
axes[0].set_xlabel('Daily Return')
axes[0].legend(fontsize=9)

# Q-Q plot
stats.probplot(tsla_ret, plot=axes[1])
axes[1].set_title(f'{FOCUS} Q-Q Plot (vs Normal)')

# Returns over time
axes[2].plot(tsla_ret.index, tsla_ret, lw=0.8, color='#1f77b4', alpha=0.7)
axes[2].axhline(0, color='black', lw=0.8, ls='--')
axes[2].fill_between(tsla_ret.index, tsla_ret, 0,
                     where=(tsla_ret >= 0), alpha=0.4, color='#2ca02c')
axes[2].fill_between(tsla_ret.index, tsla_ret, 0,
                     where=(tsla_ret < 0),  alpha=0.4, color='#d62728')
axes[2].set_title(f'{FOCUS} Daily Returns Over Time')
axes[2].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))

plt.tight_layout()
plt.show()

# Skewness / Kurtosis
sk = float(tsla_ret.skew())
kt = float(tsla_ret.kurt())
print(f'TSLA return stats: mean={mu:.4f}  σ={sigma:.4f}  skew={sk:.2f}  excess-kurt={kt:.2f}')
print(f'Fat tails: {"YES" if abs(kt) > 1 else "NO"} | Negative skew: {"YES" if sk < 0 else "NO"}')

In [ ]:
cum_returns = (1 + daily_returns).cumprod() - 1

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Cumulative returns
for t in cum_returns.columns:
    axes[0].plot(cum_returns.index, cum_returns[t], lw=1.4, label=t)
axes[0].axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].set_title('Cumulative Returns Since Jan 2024')
axes[0].set_ylabel('Total Return')
axes[0].legend(ncol=2, fontsize=9)

# Rolling 30-day annualised volatility comparison
roll_vol = daily_returns.rolling(30).std() * np.sqrt(252)
for t in roll_vol.columns:
    axes[1].plot(roll_vol.index, roll_vol[t], lw=1.2, label=t, alpha=0.85)
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_title('30-Day Rolling Annualised Volatility')
axes[1].set_ylabel('Annualised Volatility')
axes[1].legend(ncol=2, fontsize=9)

plt.tight_layout()
plt.show()

---
## 5. Risk Metrics

Computed for all stocks:
- **Sharpe Ratio** (annualised, risk-free = 0)
- **Calmar Ratio** (annualised return / max drawdown)
- **VaR 95%/99%** and **CVaR 95%** (historical)
- **Maximum Drawdown**
- **Beta** relative to SPY

In [ ]:
def compute_risk_metrics(returns: pd.Series, name: str, spy_ret: pd.Series) -> dict:
    r = returns.dropna()
    ann_ret = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0

    cum     = (1 + r).cumprod()
    peak    = cum.cummax()
    dd      = (cum - peak) / peak
    max_dd  = float(dd.min())
    calmar  = ann_ret / abs(max_dd) if max_dd != 0 else 0

    var95  = float(np.percentile(r, 5))
    var99  = float(np.percentile(r, 1))
    cvar95 = float(r[r <= var95].mean())

    # Beta vs SPY
    aligned = pd.concat([r, spy_ret], axis=1).dropna()
    if len(aligned) > 10 and aligned.columns[1] != aligned.columns[0]:
        cov = np.cov(aligned.iloc[:, 0], aligned.iloc[:, 1])
        beta = cov[0, 1] / cov[1, 1]
    else:
        beta = 1.0

    return {
        'Ticker'        : name,
        'Ann. Return'   : f'{ann_ret:+.1%}',
        'Ann. Volatility': f'{ann_vol:.1%}',
        'Sharpe'        : f'{sharpe:.2f}',
        'Calmar'        : f'{calmar:.2f}',
        'Max Drawdown'  : f'{max_dd:.1%}',
        'VaR 95%'       : f'{var95:.2%}',
        'CVaR 95%'      : f'{cvar95:.2%}',
        'VaR 99%'       : f'{var99:.2%}',
        'Beta (vs SPY)' : f'{beta:.2f}',
    }

spy_ret = daily_returns.get('SPY', pd.Series(dtype=float))
rows    = [compute_risk_metrics(daily_returns[t], t, spy_ret) for t in daily_returns.columns]
risk_df = pd.DataFrame(rows).set_index('Ticker')

print('=== Risk Metrics Summary (2024 – Present) ===')
risk_df

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

# Cumulative returns
for t in cum_returns.columns:
    axes[0].plot(cum_returns.index, cum_returns[t], lw=1.3, label=t)
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].axhline(0, color='black', ls='--', lw=0.7)
axes[0].set_title('Cumulative Return')
axes[0].legend(ncol=4, fontsize=9)

# Drawdown
for t in daily_returns.columns:
    r   = daily_returns[t].dropna()
    cum = (1 + r).cumprod()
    dd  = (cum - cum.cummax()) / cum.cummax()
    axes[1].plot(dd.index, dd, lw=1.2, label=t, alpha=0.85)
    axes[1].fill_between(dd.index, dd, 0, alpha=0.07)

axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_title('Drawdown from Peak')
axes[1].set_ylabel('Drawdown')
axes[1].legend(ncol=4, fontsize=9)

plt.tight_layout()
plt.show()

---
## 6. Multi-Stock Comparative Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Risk-Return scatter
ann_rets  = daily_returns.mean() * 252
ann_vols  = daily_returns.std() * np.sqrt(252)
sharpes   = ann_rets / ann_vols

scatter = axes[0].scatter(
    ann_vols, ann_rets,
    c=sharpes, cmap='RdYlGn', s=120, zorder=3,
    vmin=sharpes.min(), vmax=sharpes.max()
)
for t in ann_rets.index:
    axes[0].annotate(t, (ann_vols[t], ann_rets[t]),
                     textcoords='offset points', xytext=(6, 4), fontsize=10)
axes[0].axhline(0, color='black', ls='--', lw=0.8, alpha=0.5)
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[0].set_xlabel('Annualised Volatility')
axes[0].set_ylabel('Annualised Return')
axes[0].set_title('Risk-Return Profile (colour = Sharpe Ratio)')
plt.colorbar(scatter, ax=axes[0], label='Sharpe Ratio')

# Correlation heatmap
corr = daily_returns.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, ax=axes[1], annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 10},
            mask=np.zeros_like(corr, dtype=bool))   # show full matrix
axes[1].set_title('Return Correlation Matrix')

plt.tight_layout()
plt.show()

In [ ]:
# 60-day rolling correlation of each EV stock to TSLA
ev_stocks = [t for t in daily_returns.columns if t not in ('SPY', 'TSLA')]
roll_corr  = {t: daily_returns[t].rolling(60).corr(daily_returns['TSLA']) for t in ev_stocks}

fig, ax = plt.subplots(figsize=(16, 5))
for t, rc in roll_corr.items():
    ax.plot(rc.index, rc, lw=1.3, label=t)
ax.axhline(0, color='black', ls='--', lw=0.8)
ax.set_title('60-Day Rolling Correlation to TSLA')
ax.set_ylabel('Pearson Correlation')
ax.legend(ncol=3, fontsize=9)
plt.tight_layout()
plt.show()

---
## 7. Machine Learning: Next-Day Return Prediction

**Approach:**
1. **Features** — 20+ lagged / indicator features (RSI, MACD, BB%B, ATR ratio, volume ratio, price-MA ratios, lagged returns)
2. **Target** — next-day close-to-close return direction (+1 / -1)
3. **Model** — Random Forest classifier (walk-forward TimeSeriesSplit, no look-ahead bias)
4. **Metrics** — Accuracy, directional accuracy, feature importance

In [ ]:
def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Build ML feature set from indicators DataFrame."""
    d = df.copy()
    c = d['Close']

    # Lagged returns (1-5 days)
    for lag in range(1, 6):
        d[f'Ret_lag{lag}'] = d['Return'].shift(lag)

    # Momentum
    d['Mom_5']  = c.pct_change(5)
    d['Mom_10'] = c.pct_change(10)
    d['Mom_20'] = c.pct_change(20)

    # Volume ratio (today vs 20-day avg)
    d['Vol_Ratio'] = d['Volume'] / d['Volume'].rolling(20).mean()

    # ATR ratio (normalised)
    d['ATR_Ratio'] = d['ATR'] / c

    feature_cols = [
        'RSI', 'MACD', 'MACD_Signal', 'BB_Pct', 'BB_Width',
        'Ratio_SMA20', 'Ratio_SMA50', 'Vol_20',
        'Ret_lag1', 'Ret_lag2', 'Ret_lag3', 'Ret_lag4', 'Ret_lag5',
        'Mom_5', 'Mom_10', 'Mom_20',
        'Vol_Ratio', 'ATR_Ratio',
    ]

    # Target: direction of next-day return
    d['Target'] = np.sign(d['Return'].shift(-1))

    feat_df = d[feature_cols + ['Target']].dropna()
    return feat_df, feature_cols

feat_df, feature_cols = build_features(tsla)
X = feat_df[feature_cols].values
y = feat_df['Target'].values
idx = feat_df.index

print(f'Feature matrix: {X.shape}  ({len(feature_cols)} features, {len(y)} samples)')
print(f'Class balance: +1 (up) = {(y==1).mean():.1%}  |  -1 (down) = {(y==-1).mean():.1%}')

In [ ]:
# Walk-forward cross-validation (5 folds, no look-ahead bias)
tscv    = TimeSeriesSplit(n_splits=5)
scaler  = StandardScaler()
rf      = RandomForestRegressor(n_estimators=200, max_depth=6,
                                min_samples_leaf=5, random_state=42, n_jobs=-1)

fold_scores = []
all_preds   = np.full(len(y), np.nan)

for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    rf.fit(X_tr_s, y_tr)
    preds = rf.predict(X_te_s)
    all_preds[test_idx] = preds

    dir_acc = np.mean(np.sign(preds) == np.sign(y_te))
    rmse    = np.sqrt(mean_squared_error(y_te, preds))
    fold_scores.append({'fold': fold+1, 'dir_acc': dir_acc, 'rmse': rmse,
                        'train_size': len(train_idx), 'test_size': len(test_idx)})
    print(f'  Fold {fold+1}: dir-accuracy={dir_acc:.1%}  RMSE={rmse:.4f}  '
          f'(train={len(train_idx)}, test={len(test_idx)})')

scores_df = pd.DataFrame(fold_scores)
print(f'\nMean directional accuracy: {scores_df["dir_acc"].mean():.1%}')
print(f'Mean RMSE:                 {scores_df["rmse"].mean():.4f}')

# Retrain on full data for importance
rf.fit(scaler.fit_transform(X), y)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Feature importance
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)
imp.plot(kind='barh', ax=axes[0], color='#1f77b4', edgecolor='white')
axes[0].set_title('Random Forest Feature Importance')
axes[0].set_xlabel('Importance Score')

# Directional accuracy by fold
axes[1].bar(scores_df['fold'], scores_df['dir_acc'], color='#2ca02c', edgecolor='white')
axes[1].axhline(0.5, color='red', ls='--', lw=1.5, label='Random baseline (50%)')
axes[1].axhline(scores_df['dir_acc'].mean(), color='blue', ls='--', lw=1.5,
                label=f'Mean = {scores_df["dir_acc"].mean():.1%}')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].set_xlabel('Fold')
axes[1].set_ylabel('Directional Accuracy')
axes[1].set_title('Walk-Forward Directional Accuracy by Fold')
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

---
## 8. Backtesting: SMA 20/50 Crossover Strategy

**Rules:**
- **BUY** signal when SMA 20 crosses above SMA 50 (golden cross)
- **SELL** signal when SMA 20 crosses below SMA 50 (death cross)
- Long-only, fully invested, no leverage, no transaction costs

Compared against **Buy & Hold** as benchmark.

In [ ]:
def backtest_ma_crossover(df: pd.DataFrame, fast: int = 20, slow: int = 50) -> pd.DataFrame:
    bt = df[['Close', 'SMA_20', 'SMA_50', 'Return']].copy()
    bt.columns    = ['Close', f'SMA_{fast}', f'SMA_{slow}', 'Return']

    # Signal: 1 = long, 0 = flat
    bt['Signal']   = (bt[f'SMA_{fast}'] > bt[f'SMA_{slow}']).astype(int)
    bt['Position'] = bt['Signal'].shift(1)   # enter next day

    # Strategy return
    bt['Strat_Return']     = bt['Return'] * bt['Position']
    bt['Cum_BH']           = (1 + bt['Return']).cumprod() - 1
    bt['Cum_Strat']        = (1 + bt['Strat_Return']).cumprod() - 1

    # Drawdown for strategy
    strat_cum = 1 + bt['Cum_Strat']
    bt['Strat_DD'] = (strat_cum - strat_cum.cummax()) / strat_cum.cummax()

    # Trade log
    bt['Trade']    = bt['Signal'].diff().fillna(0)
    buys  = bt[bt['Trade'] ==  1].index
    sells = bt[bt['Trade'] == -1].index

    return bt, buys, sells

bt, buys, sells = backtest_ma_crossover(tsla)

# Performance
bh_total    = float(bt['Cum_BH'].iloc[-1])
strat_total = float(bt['Cum_Strat'].iloc[-1])
n_trades    = len(buys) + len(sells)

strat_ann_ret = bt['Strat_Return'].mean() * 252
strat_ann_vol = bt['Strat_Return'].std() * np.sqrt(252)
strat_sharpe  = strat_ann_ret / strat_ann_vol if strat_ann_vol > 0 else 0

bh_ann_ret = bt['Return'].mean() * 252
bh_ann_vol = bt['Return'].std() * np.sqrt(252)
bh_sharpe  = bh_ann_ret / bh_ann_vol if bh_ann_vol > 0 else 0

print('=== MA 20/50 Crossover Backtest ===')
print(f'  Number of signals:           {n_trades}')
print(f'  Buy & Hold total return:     {bh_total:+.1%}')
print(f'  Strategy total return:       {strat_total:+.1%}')
print(f'  Buy & Hold Sharpe:           {bh_sharpe:.2f}')
print(f'  Strategy Sharpe:             {strat_sharpe:.2f}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True,
                         gridspec_kw={'height_ratios': [3, 2, 2]})

# Price + signals
axes[0].plot(bt.index, bt['Close'], lw=1.3, color='#1f77b4', label='Close', zorder=2)
axes[0].plot(bt.index, bt['SMA_20'], lw=1, color='orange',  label='SMA 20')
axes[0].plot(bt.index, bt['SMA_50'], lw=1, color='green',   label='SMA 50')
axes[0].scatter(buys,  bt.loc[buys,  'Close'], marker='^', color='lime',   s=80,
                zorder=4, label='Buy signal')
axes[0].scatter(sells, bt.loc[sells, 'Close'], marker='v', color='red',    s=80,
                zorder=4, label='Sell signal')

# Shade position periods
in_position = bt['Position'] == 1
axes[0].fill_between(bt.index, bt['Close'].min(), bt['Close'].max(),
                     where=in_position, alpha=0.05, color='green', label='In position')
axes[0].set_title(f'{FOCUS} – MA 20/50 Crossover Strategy Signals')
axes[0].set_ylabel('Price (USD)')
axes[0].legend(ncol=4, fontsize=9)

# Cumulative returns comparison
axes[1].plot(bt.index, bt['Cum_BH']    * 100, lw=1.5, color='#1f77b4', label=f'Buy & Hold ({bh_total:+.1%})')
axes[1].plot(bt.index, bt['Cum_Strat'] * 100, lw=1.5, color='#2ca02c', label=f'MA Crossover ({strat_total:+.1%})')
axes[1].axhline(0, color='black', ls='--', lw=0.8)
axes[1].set_title('Strategy vs Buy & Hold — Cumulative Return')
axes[1].set_ylabel('Return (%)')
axes[1].legend(fontsize=9)

# Strategy drawdown
axes[2].fill_between(bt.index, bt['Strat_DD'] * 100, 0, alpha=0.4, color='#d62728')
axes[2].plot(bt.index, bt['Strat_DD'] * 100, lw=1, color='#d62728')
axes[2].set_title('Strategy Drawdown')
axes[2].set_ylabel('Drawdown (%)')

plt.tight_layout()
plt.show()

---
## 9. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 24))
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.5, wspace=0.35)

# 1. Normalised performance
ax0 = fig.add_subplot(gs[0, :])
norm = close.div(close.iloc[0]) * 100
for t in norm.columns:
    ax0.plot(norm.index, norm[t], lw=1.6, label=t)
ax0.axhline(100, color='black', ls='--', lw=0.7, alpha=0.4)
ax0.set_title('Normalised Price Performance (Base = 100) — Jan 2024 to Present', fontsize=14)
ax0.set_ylabel('Indexed Price')
ax0.legend(ncol=4, fontsize=10)

# 2. Annualised returns bar chart
ax1 = fig.add_subplot(gs[1, 0])
colors = ['#2ca02c' if r >= 0 else '#d62728' for r in ann_rets]
ann_rets.plot(kind='bar', ax=ax1, color=colors, edgecolor='white')
ax1.axhline(0, color='black', lw=0.8)
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax1.set_title('Annualised Return')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=0)

# 3. Volatility bar chart
ax2 = fig.add_subplot(gs[1, 1])
ann_vols.plot(kind='bar', ax=ax2, color='#1f77b4', edgecolor='white', alpha=0.8)
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.set_title('Annualised Volatility')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=0)

# 4. TSLA Price + SMA
ax3 = fig.add_subplot(gs[2, :])
ax3.plot(tsla.index, tsla['Close'],   lw=1.5, color='#1f77b4', label='Close')
ax3.plot(tsla.index, tsla['SMA_20'],  lw=1,   color='orange',  label='SMA 20')
ax3.plot(tsla.index, tsla['SMA_50'],  lw=1,   color='green',   label='SMA 50')
ax3.plot(tsla.index, tsla['SMA_200'], lw=1,   color='red',     label='SMA 200')
ax3.fill_between(tsla.index, tsla['BB_Lower'], tsla['BB_Upper'],
                 alpha=0.1, color='purple', label='Bollinger Bands')
ax3.set_title(f'{FOCUS} Price + MAs + Bollinger Bands', fontsize=13)
ax3.set_ylabel('Price (USD)')
ax3.legend(ncol=5, fontsize=9)

# 5. Correlation heatmap
ax4 = fig.add_subplot(gs[3, 0])
sns.heatmap(daily_returns.corr(), ax=ax4, annot=True, fmt='.2f',
            cmap='RdYlGn', vmin=-1, vmax=1, linewidths=0.5, annot_kws={'size': 9})
ax4.set_title('Return Correlation Matrix')

# 6. Returns distribution
ax5 = fig.add_subplot(gs[3, 1])
ax5.hist(tsla_ret, bins=65, density=True, color='#1f77b4', edgecolor='white', alpha=0.8)
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
ax5.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', lw=2, label='Normal')
ax5.axvline(np.percentile(tsla_ret, 5), color='orange', ls='--', lw=1.5, label='VaR 95%')
ax5.set_title(f'{FOCUS} Daily Returns Distribution')
ax5.set_xlabel('Daily Return')
ax5.legend(fontsize=9)

fig.suptitle(
    f'Advanced Stock Market Analysis Dashboard  |  {START_DATE} → {END_DATE}',
    fontsize=16, fontweight='bold'
)
plt.savefig('analysis_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dashboard saved → analysis_dashboard.png')

In [ ]:
print('=' * 65)
print(f'  STOCK MARKET ANALYSIS — FINAL SUMMARY  ({START_DATE} → {END_DATE})')
print('=' * 65)
print(risk_df.to_string())
print('\n--- TSLA Backtest: MA 20/50 Crossover ---')
print(f'  Buy & Hold return :  {bh_total:+.1%}')
print(f'  MA Strategy return:  {strat_total:+.1%}')
print(f'  Buy & Hold Sharpe :  {bh_sharpe:.2f}')
print(f'  MA Strategy Sharpe:  {strat_sharpe:.2f}')
print(f'  Total signals     :  {n_trades}')
print('\n--- ML Directional Prediction (Random Forest) ---')
print(f'  Mean directional accuracy: {scores_df["dir_acc"].mean():.1%}')
print(f'  (5-fold walk-forward CV, {len(feature_cols)} features)')
print('=' * 65)

---
## Notes & Limitations

- **Data:** All prices are adjusted for splits/dividends via `auto_adjust=True`.
- **Risk-free rate:** Set to 0 for simplicity; Sharpe ratios would be lower vs a risk-free benchmark.
- **Backtest:** No transaction costs, slippage, or taxes modelled. Real performance would differ.
- **ML model:** Directional prediction is a classification proxy; real trading systems require further validation.
- **Data range:** All analysis is live — re-run the notebook at any time to refresh with the latest data.